# 01b — Configuration & Exposure Module Demo Figures

Reproduces the two showcase figures used in the CAT411 slide deck and in §2.2 / §2.4 of the final report:

1. **Figure 2.2** — How four different YAML configurations slice the same exposure database into different portfolios.
2. **Figure 2.3** — What the cached California 2024 NBI database contains (25,848 bridges, $168 B RCV).

Both figures read from the cached workbook `output/exposure/nbi_all_bridges_rcv.xlsx`, which is built once by `scripts/compute_rcv_and_vs30.py` and contains one row per bridge with lat/lon, material, year built, deck area, max span, skew, county FIPS, Vs30, NEHRP class, HWB class, and FHWA 2024 replacement cost.

**Run order:** `Run → Run All Cells`. The two figures land in `output/exposure/config_demo.png` and `output/exposure/exposure_demo.png`. The notebook is self-contained — no prior notebook execution required.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook lives in tutorials/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'tutorials' else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'exposure'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(OUTPUT_DIR / 'nbi_all_bridges_rcv.xlsx')
print(f'Loaded {len(df):,} bridges, total RCV ${df.replacement_cost_usd.sum()/1e9:.1f} B')
df.head()

## Figure 2.2 — Four configurations applied to the same database

Each panel uses identical plotting code; only the YAML filter differs.

| Config | Filter | Bridges |
|---|---|---|
| A. Statewide | none | 25,848 |
| B. Northridge bbox | lat 33.8–34.6, lon −118.9 to −118.0 | 2,953 |
| C. LA County, concrete only | county = 37 (LA), material = concrete | 2,341 |
| D. Pre-1975 conventional | year_built < 1975 | 16,706 |

Marker size encodes per-bridge replacement cost; color encodes material.

In [ ]:
mat_colors = {
    'concrete':             '#4C72B0',
    'prestressed_concrete': '#55A868',
    'steel':                '#C44E52',
    'wood':                 '#8C613C',
    'masonry':              '#937860',
    'aluminum_iron':        '#CCB974',
    'other':                '#999999',
}
plot_order = list(mat_colors.keys())

configs = {
    'A. Statewide (all CA)': df.copy(),
    'B. Northridge bbox\n(33.8-34.6, -118.9 to -118.0)': df[
        (df.latitude.between(33.8, 34.6)) & (df.longitude.between(-118.9, -118.0))
    ].copy(),
    'C. LA County + concrete only': df[
        (df.county == 37) & (df.material == 'concrete')
    ].copy(),
    'D. Pre-1975 conventional design\n(higher seismic risk)': df[
        df.year_built < 1975
    ].copy(),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(
    'Configuration — How Different Filters Slice the Same Database',
    fontsize=14, fontweight='bold', y=0.995,
)

for ax, (name, sub) in zip(axes.flat, configs.items()):
    if len(sub) == 0:
        ax.text(0.5, 0.5, 'No bridges', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(name, fontsize=11, fontweight='bold')
        continue

    for mat in plot_order:
        sel = sub[sub.material == mat]
        if len(sel) == 0:
            continue
        size = np.clip(np.sqrt(sel.replacement_cost_usd / 1e6) * 1.5, 1, 60)
        ax.scatter(sel.longitude, sel.latitude,
                   c=mat_colors[mat], s=size, alpha=0.55,
                   edgecolors='none', label=f'{mat} (n={len(sel)})')

    info = (f'n = {len(sub):,} bridges\n'
            f'Total RCV: ${sub.replacement_cost_usd.sum()/1e9:.1f}B\n'
            f'Avg Vs30: {sub.vs30.mean():.0f} m/s\n'
            f'Avg year built: {sub.year_built.mean():.0f}')
    ax.text(0.02, 0.98, info, transform=ax.transAxes, fontsize=9,
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='gray', alpha=0.85))

    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

    if 'Statewide' in name or 'Pre-1975' in name:
        ax.set_xlim([-125, -113.5]); ax.set_ylim([32, 42.5])
    elif 'Northridge' in name:
        ax.set_xlim([-119.0, -117.9]); ax.set_ylim([33.7, 34.7])
    else:
        ax.set_xlim([-118.95, -117.6]); ax.set_ylim([33.7, 34.85])

    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='lower right', fontsize=7, framealpha=0.85, markerscale=0.7)

plt.tight_layout()
fig1_path = OUTPUT_DIR / 'config_demo.png'
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig1_path}')

## Figure 2.3 — What the exposure database contains

Four panels summarising the cached California 2024 NBI build:

1. **Spatial distribution by material** — every bridge plotted at its lat/lon, coloured by material.
2. **Replacement-cost intensity** — log-scale heatmap showing where the high-RCV bridges concentrate.
3. **Year-built histogram** — pre-1975 cohort highlighted (modern seismic provisions introduced).
4. **NEHRP site-class distribution** — bridges binned by NEHRP class (B = rock → E = soft soil).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle(
    f'Exposure Database — California 2024 NBI ({len(df):,} bridges, '
    f'${df.replacement_cost_usd.sum()/1e9:.0f}B RCV)',
    fontsize=14, fontweight='bold', y=0.995,
)

# Panel 1: Statewide map by material
ax = axes[0, 0]
for mat in plot_order:
    sel = df[df.material == mat]
    if len(sel) == 0:
        continue
    ax.scatter(sel.longitude, sel.latitude,
               c=mat_colors[mat], s=2, alpha=0.4, edgecolors='none',
               label=f'{mat} (n={len(sel):,})')
ax.set_title('a) Spatial distribution by material', fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_xlim([-125, -113.5]); ax.set_ylim([32, 42.5])
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='lower right', fontsize=7, framealpha=0.85, markerscale=3)

# Panel 2: RCV intensity (log scale)
ax = axes[0, 1]
log_rcv = np.log10(df.replacement_cost_usd.clip(lower=1e4))
sc = ax.scatter(df.longitude, df.latitude, c=log_rcv, s=3,
                cmap='plasma', alpha=0.6, edgecolors='none',
                vmin=5, vmax=9)
cb = plt.colorbar(sc, ax=ax, shrink=0.8)
cb.set_label('log10(RCV in USD)', fontsize=9)
ax.set_title('b) Replacement cost intensity', fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_xlim([-125, -113.5]); ax.set_ylim([32, 42.5])
ax.grid(True, alpha=0.3, linestyle='--')

# Panel 3: Year-built histogram
ax = axes[1, 0]
ax.hist(df.year_built.dropna(), bins=np.arange(1860, 2030, 5),
        color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(1975, color='red', linestyle='--', linewidth=1.5,
           label='1975 — modern seismic code')
ax.set_title('c) Year built distribution', fontsize=11, fontweight='bold')
ax.set_xlabel('Year built'); ax.set_ylabel('Number of bridges')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, linestyle='--', axis='y')
pre = (df.year_built < 1975).sum()
post = (df.year_built >= 1975).sum()
ax.text(0.02, 0.95,
        f'Pre-1975: {pre:,} ({pre/len(df)*100:.0f}%)\n'
        f'Post-1975: {post:,} ({post/len(df)*100:.0f}%)',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='gray', alpha=0.85))

# Panel 4: NEHRP site class distribution
ax = axes[1, 1]
nehrp_order = ['B', 'BC', 'C', 'CD', 'D', 'E']
counts = [(df.nehrp_class == c).sum() for c in nehrp_order]
colors_nehrp = ['#08306B', '#2171B5', '#6BAED6', '#FDAE61', '#D7301F', '#7F0000']
bars = ax.bar(nehrp_order, counts, color=colors_nehrp, edgecolor='white')
ax.set_title('d) Site condition (NEHRP class from Vs30)', fontsize=11, fontweight='bold')
ax.set_xlabel('NEHRP site class (B=rock → E=soft soil)')
ax.set_ylabel('Number of bridges')
ax.grid(True, alpha=0.3, linestyle='--', axis='y')
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{n:,}', ha='center', fontsize=9)
ax.text(0.98, 0.95,
        f'Avg Vs30: {df.vs30.mean():.0f} m/s\n'
        f'Median: {df.vs30.median():.0f} m/s',
        transform=ax.transAxes, fontsize=9, va='top', ha='right',
        bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='gray', alpha=0.85))

plt.tight_layout()
fig2_path = OUTPUT_DIR / 'exposure_demo.png'
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fig2_path}')

## Portfolio statistics summary

Headline numbers used in §2.4 of the report — useful as a one-line summary in any downstream notebook.

In [ ]:
summary = pd.Series({
    'Total bridges':           f'{len(df):,}',
    'Total RCV':               f'${df.replacement_cost_usd.sum()/1e9:.1f} B',
    'Average RCV per bridge':  f'${df.replacement_cost_usd.mean()/1e6:.1f} M',
    'Median RCV':              f'${df.replacement_cost_usd.median()/1e6:.1f} M',
    'P90 RCV':                 f'${df.replacement_cost_usd.quantile(0.9)/1e6:.1f} M',
    'Average Vs30':            f'{df.vs30.mean():.0f} m/s',
    'Pre-1975 share':          f'{(df.year_built < 1975).mean()*100:.0f}%',
    'Concrete share':          f'{(df.material == "concrete").mean()*100:.0f}%',
    'Prestressed share':       f'{(df.material == "prestressed_concrete").mean()*100:.0f}%',
    'Steel share':             f'{(df.material == "steel").mean()*100:.0f}%',
})
summary.to_frame('Value')

## Try it: define your own configuration

Edit the filter expression in the cell below to slice the database your own way. The plot updates automatically on `Run`.

In [ ]:
# Example: prestressed concrete bridges within 50 km of downtown San Francisco
lat_sf, lon_sf = 37.7749, -122.4194
deg_radius = 0.5   # roughly 50 km at this latitude

custom = df[
    (df.material == 'prestressed_concrete')
    & (df.latitude.between(lat_sf - deg_radius, lat_sf + deg_radius))
    & (df.longitude.between(lon_sf - deg_radius, lon_sf + deg_radius))
]

print(f'Custom filter: {len(custom):,} bridges, '
      f'total RCV ${custom.replacement_cost_usd.sum()/1e9:.2f} B')

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(custom.longitude, custom.latitude,
           c='#55A868', s=np.clip(np.sqrt(custom.replacement_cost_usd/1e6)*2, 4, 80),
           alpha=0.6, edgecolors='none')
ax.scatter([lon_sf], [lat_sf], marker='*', s=300, c='red',
           edgecolors='black', zorder=5, label='SF downtown')
ax.set_title(f'Custom config — prestressed concrete near SF (n={len(custom):,})',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_xlim([lon_sf - deg_radius, lon_sf + deg_radius])
ax.set_ylim([lat_sf - deg_radius, lat_sf + deg_radius])
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='lower right')
plt.show()

---

**What's next:** see `02_hazard_shakemap.ipynb` for how Sa(1.0 s) is interpolated to each bridge site, and `06_loss_and_cost.ipynb` for how RCV is combined with damage probabilities to compute expected loss.